In [72]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

# Load Vectorstore

In [73]:
EMBEDDING_MODEL = 'hf.co/Casual-Autopsy/snowflake-arctic-embed-l-v2.0-gguf:Q4_K_M'
OLLAMA_EMBEDDING_BASE_URL = 'http://127.0.0.1:11434'
persist_directory = "../chroma_db"

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_EMBEDDING_BASE_URL,
)

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
)

In [74]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x141fd3a50>, search_kwargs={'k': 5})

# Prompt


In [75]:
from langchain_core.prompts import ChatPromptTemplate

In [76]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Ты помощник, который ОТВЕЧАЕТ СТРОГО НА РУССКОМ ЯЗЫКЕ. "
        "Даже если контекст или вопрос частично на других языках (английский, немецкий и т.п.), "
        "ты ВСЁ РАВНО отвечаешь только на русском, используя кириллицу. "
        "Не используй слова и фразы на других языках, кроме общепринятых имен собственных и названий. "
        "Используй следующие фрагменты контекста для ответа на вопрос. "
        "Если ты не знаешь ответа, просто скажи, что не знаешь. "
        "Используй максимум три предложения и будь лаконичным."
            ),
    (
        "human",
        "Контекст: {context}\n\nВопрос: {question}"
    ),
])

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Ты помощник, который ОТВЕЧАЕТ СТРОГО НА РУССКОМ ЯЗЫКЕ. Даже если контекст или вопрос частично на других языках (английский, немецкий и т.п.), ты ВСЁ РАВНО отвечаешь только на русском, используя кириллицу. Не используй слова и фразы на других языках, кроме общепринятых имен собственных и названий. Используй следующие фрагменты контекста для ответа на вопрос. Если ты не знаешь ответа, просто скажи, что не знаешь. Используй максимум три предложения и будь лаконичным.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Контекст: {context}\n\nВопрос: {question}'), additional_kwargs={})])

# llm

In [77]:
from langchain_openai import ChatOpenAI

In [78]:
OLLAMA_BASE_URL = 'http://127.0.0.1:11434/v1'
LLM_MODEL = 'gemma3:4b'

llm = ChatOpenAI(
    api_key='None',
    base_url=OLLAMA_BASE_URL,
    model=LLM_MODEL,
)

In [67]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x14c39d890>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x14c39fd90>, root_client=<openai.OpenAI object at 0x14c39d850>, root_async_client=<openai.AsyncOpenAI object at 0x14c39cd90>, model_name='gemma3:4b', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://127.0.0.1:11434/v1')

# LangChain

In [81]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [82]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Request

In [83]:
rag_chain.invoke("Какие услуги есть у компании A&K?")

'Компания A&K предлагает услуги по оформлению студенческих и туристических виз в США, а также шенгенских виз. Кроме того, компания предоставляет консультации по оценке шансов на получение визы, языковые курсы и помощь в подборе билетов и проживания в США.'

In [84]:
rag_chain.invoke("Вы можете помочь с лотереей Greed Card?")

'Да, мы можем помочь вам заполнить анкету и проверить ее на наличие ошибок, чтобы увеличить ваши шансы на победу в лотерее Green Card DV-2027. Участие в этой лотерее стоит $35.'

In [85]:
 rag_chain.invoke("Сколько стоит виза?")

'Стоимость виз варьируется в зависимости от пакета сопровождения. Базовое сопровождение стоит $2 200 с первым взносом при оплате частями, Премиум сопровождение - $790, а Полное сопровождение - от $1 190. Также доступны отдельные услуги, такие как консультация визового эксперта за $149.'